# Predicta — Sprint 4: Integração da Solução e Suporte Operacional (RPA)

**Equipe**

| Nome | RM |
|---|---|
| Sofia Bueris Netto de Souza | 565818 |
| Augusto Oliveira Codo de Sousa | 562080 |
| Gabriel Tonelli Avelino Dos Santos | 564705 |
| Vinícius Adrian Siqueira de Oliveira | 564962 |
| Felipe de Oliveira Cabral | 561720 |

Este notebook executa o pipeline integrado de ponta a ponta: **alerta detectado → status atualizado → recomendação técnica registrada → encaminhamento → encerramento**, e mostra as evidências.

Roteiro: 1) preparar o ambiente, 2) rodar a demonstração (14 ciclos), 3) conferir o fluxo de um evento, 4) ver o log estruturado, 5) ver o que o assistente recebe, 6) validar a integridade, 7) rodar os testes.

## 1. Preparar o ambiente

O notebook procura o projeto na pasta atual. Se não encontrar, baixa do GitHub (preencha `REPO_URL`) ou pede o `.zip` do projeto.

In [1]:
import os, sys, subprocess, pathlib

REPO_URL = ""   # ex.: "https://github.com/<usuario>/predicta-digital-twin.git"

def achar_raiz():
    for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (p / "src" / "integracao").exists():
            return p
    return None

raiz = achar_raiz()
if raiz is None and REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "predicta"], check=True)
    raiz = pathlib.Path("predicta").resolve()
if raiz is None:
    try:
        from google.colab import files
        print("Envie o arquivo .zip do projeto:")
        enviado = files.upload()
        import zipfile
        nome = next(iter(enviado))
        zipfile.ZipFile(nome).extractall("predicta_zip")
        raiz = next(p.parent.parent for p in pathlib.Path("predicta_zip").rglob("orquestrador.py"))
    except ImportError:
        raise SystemExit("Projeto não encontrado: abra o notebook dentro da pasta do projeto ou preencha REPO_URL.")

os.chdir(raiz)
sys.path.insert(0, str(raiz))
print("Projeto em:", raiz)

Projeto em: /home/claude/predicta


In [2]:
# Dependências mínimas. IA generativa (Ollama) e FAISS são opcionais: sem eles o pipeline usa regras + busca léxica.
cmd = [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-core.txt"]
if subprocess.run(cmd, capture_output=True).returncode != 0:     # ambientes com Python "gerenciado" exigem esta opção
    subprocess.run(cmd + ["--break-system-packages"], check=True, capture_output=True)
print("Dependências prontas.")

Dependências prontas.


## 2. Executar a demonstração integrada

Recria o banco e roda a carga histórica (ciclo 0) e 14 ciclos com um roteiro fixo. Entre os ciclos, o roteiro inclui atendimento de um técnico e falhas injetadas (arquivo corrompido, motor não cadastrado, sensor sem sinal e uma etapa que falha), para provar a resiliência.

In [3]:
import logging
logging.basicConfig(level=logging.WARNING, format="%(message)s")
logging.getLogger("src").setLevel(logging.ERROR)   # silencia o detalhe das sprints anteriores; o log completo está em evidencias/logs
from src.integracao.demo import executar_demo

resultado = executar_demo(ciclos=14, intervalo=0)
print("run_id:", resultado["run_id"])
print(f"{'ciclo':>5}  {'status':<8}  etapas com ressalva")
for r in resultado["ciclos"]:
    prob = ", ".join(f"{e.nome}:{e.status}" for e in r.etapas if e.status != "ok") or "-"
    print(f"{r.ciclo:>5}  {r.status:<8}  {prob}")

Erro ao buscar API Open-Meteo: HTTPSConnectionPool(host='api.open-meteo.com', port=443): Max retries exceeded with url: /v1/forecast?latitude=-23.5505&longitude=-46.6333&current_weather=true (Caused by NameResolutionError("HTTPSConnection(host='api.open-meteo.com', port=443): Failed to resolve 'api.open-meteo.com' ([Errno -2] Name or service not known)"))


[ciclo 0] catalogo/clima_indisponivel API de clima indisponível; seguindo sem contexto climático


[ciclo 0] catalogo/fim Etapa 'catalogo' terminou com status degradado


[ciclo 0] ciclo/fim Ciclo 0 terminou: parcial


[ciclo 6] coleta/arquivo_em_quarentena leituras_MTR-003.csv não pôde ser processado (sem_registros_validos); movido para quarentena


Motor 'MTR-099' não cadastrado no banco. Cadastre antes de processar.


... (6 outputs left out)

[ciclo 7] analise/fim Etapa 'analise' terminou com status falha


[ciclo 7] ciclo/fim Ciclo 7 terminou: parcial


[ciclo 8] avaliacao/sem_leitura_nova (MTR-ENV-TRP-005) Nenhuma leitura nova; ativo mantido como está


run_id: run_20261105_232216_bca013
ciclo  status    etapas com ressalva
    0  parcial   catalogo:degradado
    1  sucesso   -
    2  sucesso   -
    3  sucesso   -
    4  sucesso   -
    5  sucesso   -
    6  parcial   coleta:degradado
    7  parcial   analise:falha
    8  sucesso   -
    9  sucesso   -
   10  sucesso   -
   11  sucesso   -
   12  sucesso   -
   13  sucesso   -
   14  sucesso   -


## 3. Fluxo ponta a ponta de cada evento

Uma linha por evento, do alerta ao encerramento.

In [4]:
import pandas as pd
pd.set_option("display.max_colwidth", 140, "display.width", 200)
trilha = pd.read_csv("evidencias/trilha_ponta_a_ponta.csv")
trilha[["evento_id","tag","5_severidade_final","2_mudancas_status_ativo","3_origem_recomendacao","4_ordem_manutencao","5_situacao_final","5_encerrado_por"]]

,evento_id,tag,5_severidade_final,2_mudancas_status_ativo,3_origem_recomendacao,4_ordem_manutencao,5_situacao_final,5_encerrado_por
0,1,MTR-L01-BOM-001,CRITICO,operacional→critico; critico→manutencao; manutencao→operacional,regras+rag,"#1 (corretiva, urgente)",resolvido,tecnico:tecnico.turno_a
1,2,MTR-L01-EST-002,ALERTA,operacional→atencao; atencao→operacional,regras+rag,"#2 (preventiva, media)",resolvido,RPA_PREDICTA
2,3,MTR-UTIL-EXA-004,CRITICO,operacional→atencao; atencao→critico; critico→operacional,regras+rag,"#3 (corretiva, urgente)",resolvido,RPA_PREDICTA
3,4,MTR-ENV-TRP-005,ALERTA,operacional→atencao; atencao→operacional,regras+rag,"#4 (preventiva, media)",resolvido,RPA_PREDICTA


In [5]:
ev = trilha.iloc[0]
print(f"EVENTO {ev.evento_id} — {ev.tag}\n")
print("1. ALERTA DETECTADO :", ev["1_alerta_detectado_em"])
print("   Leitura          :", ev["1_leitura_que_motivou_classificacao"])
print("   Descrição        :", ev["1_descricao_alerta"])
print("2. STATUS DO ATIVO  :", ev["2_mudancas_status_ativo"])
print("3. RECOMENDAÇÃO     :", ev["3_recomendacao_acao"])
print("   Origem           :", ev["3_origem_recomendacao"], f"({ev['3_qtd_procedimentos']} procedimentos, {ev['3_qtd_referencias_manual']} referências)")
print("4. ENCAMINHAMENTO   :", ev["4_ordem_manutencao"], "->", ev["4_encaminhado_para"])
print("5. ENCERRAMENTO     :", ev["5_encerrado_em"], "por", ev["5_encerrado_por"])
print("   Resolução        :", ev["5_resolucao"])
print("\nLinha do tempo:", ev["transicoes"])

EVENTO 1 — MTR-L01-BOM-001

1. ALERTA DETECTADO : 2026-11-05 23:22:17
   Leitura          : 2026-08-25 06:30:00 | 88.11 °C | 5.03 mm/s | 15.07 A | 384.72 V
   Descrição        : Temperatura 88.1°C excede o limite de 85°C. Vibração 5.03 mm/s excede o limite de 2.8 mm/s (zona D da ISO 10816-3).
2. STATUS DO ATIVO  : operacional→critico; critico→manutencao; manutencao→operacional
3. RECOMENDAÇÃO     : Atuar com prioridade: Parar a máquina assim que a operação permitir, drenar a graxa antiga, relubrificar os mancais e conferir o alinhamento a laser. Realizar alinhamento a laser do eixo com a carga e inspecionar o acoplamento.
   Origem           : regras+rag (9 procedimentos, 4 referências)
4. ENCAMINHAMENTO   : #1 (corretiva, urgente) -> Equipe de Manutenção
5. ENCERRAMENTO     : 2026-11-05 23:22:17 por tecnico:tecnico.turno_a
   Resolução        : Resolvido por tecnico.turno_a: Realinhado o eixo a laser e trocado o rolamento do lado acoplado; vibração normalizada.

Linha do tempo: abertu

In [6]:
import sqlite3, json
con = sqlite3.connect("evidencias/predicta_evidencia.db")
procs = json.loads(con.execute("SELECT procedimentos_json FROM eventos_operacionais WHERE id=1").fetchone()[0])
pd.DataFrame(procs)[["ordem","tipo","descricao","referencia"]]

,ordem,tipo,descricao,referencia
0,1,corretivo,"Parar a máquina assim que a operação permitir, drenar a graxa antiga, relubrificar os mancais e conferir o alinhamento a laser.",manual_normativo_motores_weg_iso.md > 3. Diretrizes de Diagnóstico Integrado
1,2,corretivo,Realizar alinhamento a laser do eixo com a carga e inspecionar o acoplamento.,manual_normativo_motores_weg_iso.md > Manutenção Corretiva
2,3,corretivo,Conferir calços de nivelamento e a fixação dos parafusos da base (pé manco).,guia_weg.txt > SINTOMA 2
3,4,corretivo,"Verificar folga nos rolamentos; se confirmada, substituir por rolamento homologado (SKF ou FAG).",guia_weg.txt > SINTOMA 2
4,5,preventivo,Incluir o ativo na análise de vibração espectral mensal.,manual_normativo_motores_weg_iso.md > Manutenção Preditiva
5,6,corretivo,Reduzir a carga operacional e confirmar se a corrente está acima da nominal da placa.,manual_normativo_motores_weg_iso.md > Manutenção Corretiva
6,7,corretivo,Inspecionar aletas e entrada de ar da tampa defletora; limpar com ar comprimido se houver sujeira.,guia_weg.txt > SINTOMA 3
7,8,corretivo,Medir a resistência de isolamento do estator (Megger) e checar desbalanceamento de tensão.,manual_normativo_motores_weg_iso.md > Manutenção Corretiva
8,9,preventivo,Programar termografia quinzenal do estator e dos mancais.,manual_normativo_motores_weg_iso.md > Manutenção Preditiva


### Histórico completo de transições (rastreabilidade)

In [7]:
pd.read_sql_query("""SELECT h.criado_em, e.tag_ativo AS tag, h.acao, h.status_novo, h.ator, h.status_ativo_anterior AS ativo_de, h.status_ativo_novo AS ativo_para
                     FROM historico_eventos h JOIN eventos_operacionais e ON e.id=h.evento_id ORDER BY h.id""", con)

,criado_em,tag,acao,status_novo,ator,ativo_de,ativo_para
0,2026-11-05 23:22:17,MTR-L01-BOM-001,abertura,pendente,RPA_PREDICTA,operacional,critico
1,2026-11-05 23:22:17,MTR-L01-BOM-001,encaminhamento,encaminhado,RPA_PREDICTA,NaN,NaN
2,2026-11-05 23:22:17,MTR-L01-EST-002,abertura,pendente,RPA_PREDICTA,operacional,atencao
3,2026-11-05 23:22:17,MTR-L01-EST-002,encaminhamento,encaminhado,RPA_PREDICTA,NaN,NaN
4,2026-11-05 23:22:17,MTR-L01-BOM-001,atendimento,em_atendimento,tecnico:tecnico.turno_a,critico,manutencao
5,2026-11-05 23:22:17,MTR-UTIL-EXA-004,abertura,pendente,RPA_PREDICTA,operacional,atencao
6,2026-11-05 23:22:17,MTR-UTIL-EXA-004,encaminhamento,encaminhado,RPA_PREDICTA,NaN,NaN
7,2026-11-05 23:22:17,MTR-L01-BOM-001,resolucao,resolvido,tecnico:tecnico.turno_a,manutencao,operacional
8,2026-11-05 23:22:17,MTR-L01-EST-002,resolucao,resolvido,RPA_PREDICTA,atencao,operacional
9,2026-11-05 23:22:17,MTR-UTIL-EXA-004,escalonamento,encaminhado,RPA_PREDICTA,atencao,critico


### Status final dos ativos

In [8]:
pd.read_csv("evidencias/status_final_ativos.csv")

,tag,motor,status_operacional,origem_atualizacao,data_atualizacao,temperatura,vibracao,ultima_leitura
0,MTR-ENV-TRP-005,MTR-005,operacional,RPA_PREDICTA,2026-11-05 23:22:18,65.92,1.22,2026-11-05 23:22:00
1,MTR-L01-BOM-001,MTR-001,operacional,tecnico:tecnico.turno_a,2026-11-05 23:22:17,72.79,1.75,2026-11-05 23:22:00
2,MTR-L01-EST-002,MTR-002,operacional,RPA_PREDICTA,2026-11-05 23:22:17,70.60,1.77,2026-11-05 23:22:00
3,MTR-L02-CMP-003,MTR-003,manutencao,rpa_integrado_cadastro,2026-11-05 23:22:16,64.06,1.18,2026-11-05 23:22:00
4,MTR-UTIL-EXA-004,MTR-004,operacional,RPA_PREDICTA,2026-11-05 23:22:17,64.19,1.20,2026-11-05 23:22:00


## 4. Log estruturado (JSON Lines)

Cada linha é um JSON independente, com `run_id`, ciclo, etapa, evento e ativo. Abaixo, o caminho completo do MTR-L01-BOM-001 no ciclo 0.

In [9]:
import json
linhas = [json.loads(l) for l in open("evidencias/logs/execucao.jsonl", encoding="utf-8")]
print(len(linhas), "registros no log")
for l in linhas:
    if l.get("ativo") == "MTR-L01-BOM-001" and l.get("ciclo") == 0:
        print(l["ts"][11:23], l["nivel"], f"{l['etapa']}/{l['evento']}", "-", l["mensagem"][:110])

379 registros no log
23:22:17.223 INFO avaliacao/leitura_classificada - CRITICO: Temperatura 88.1°C excede o limite de 85°C. Vibração 5.03 mm/s excede o limite de 2.8 mm/s (zona D da
23:22:17.225 INFO suporte/recomendacao_gerada - Recomendação gerada (regras+rag)
23:22:17.226 INFO status/status_atualizado - operacional -> critico
23:22:17.226 INFO evento/evento_aberto - Evento #1 aberto e encaminhado (OS #1)
23:22:17.229 INFO notificacao/alerta_enviado - Alerta enviado à equipe


### Exceções tratadas durante a execução

In [10]:
df = pd.DataFrame(linhas)
df[df.nivel.isin(["WARNING", "ERROR"])][["ciclo", "nivel", "etapa", "evento", "mensagem"]].head(20)

,ciclo,nivel,etapa,evento,mensagem
10,0.0,WARNING,catalogo,clima_indisponivel,API de clima indisponível; seguindo sem contexto climático
11,0.0,WARNING,catalogo,fim,Etapa 'catalogo' terminou com status degradado
31,0.0,WARNING,ciclo,fim,Ciclo 0 terminou: parcial
165,6.0,WARNING,coleta,arquivo_em_quarentena,leituras_MTR-003.csv não pôde ser processado (sem_registros_validos); movido para quarentena
166,6.0,WARNING,coleta,arquivo_em_quarentena,leituras_MTR-099.csv não pôde ser processado (motor_nao_cadastrado); movido para quarentena
167,6.0,WARNING,coleta,fim,Etapa 'coleta' terminou com status degradado
176,6.0,WARNING,avaliacao,sem_leitura_nova,Nenhuma leitura nova; ativo mantido como está
180,6.0,WARNING,evento,evento_escalonado,Evento #3: ALERTA -> CRITICO
185,6.0,WARNING,ciclo,fim,Ciclo 6 terminou: parcial
198,7.0,ERROR,analise,excecao,Etapa 'analise' falhou: RuntimeError: falha simulada na etapa 'analise' (teste de resiliência)


## 5. O que o assistente recebe (RAG)

A cada ciclo, o estado atual e o histórico de cada ativo são exportados e indexados junto com os manuais.

In [11]:
from src.integracao.contexto_rag import montar_contexto_assistente
print(montar_contexto_assistente("Qual o histórico do MTR-001 e o que foi recomendado?", top_k=1)[:1800])

# Ativo MTR-L01-BOM-001 (MTR-001)

## Perfil
Motor WEG W22 IE3 Premium na Planta Produto Alpha, Zona Linha 1, Secao Bombeamento (Bombas linha 1 - pedestal A).
Placa: potência 15.0 kW, tensão nominal 380.0 V, corrente nominal 28.5 A, 3600.0 rpm.
Limites usados: temperatura máxima 85 °C e vibração máxima 2.8 mm/s (fonte: catalogo).

## Estado atual
Status operacional: operacional.
Última leitura (2026-11-05 23:22:00): temperatura 72.79 °C, vibração 1.75 mm/s, corrente 14.76 A, tensão 377.91 V, 1751.0 rpm.
Últimas 100 leituras: temperatura média 76.7 °C (máx. 89.8), vibração média 2.59 mm/s (máx. 5.18).

## Histórico de eventos
Evento 1 (2026-11-05 23:22:17) - CRITICO, situação resolvido, 5 ocorrência(s). Temperatura 88.1°C excede o limite de 85°C. Vibração 5.03 mm/s excede o limite de 2.8 mm/s (zona D da ISO 10816-3).
Diagnóstico: Ativo MTR-L01-BOM-001 (WEG W22 IE3 Premium): Temperatura 88.1°C excede o limite de 85°C. Vibração 5.03 mm/s excede o limite de 2.8 mm/s (zona D da ISO 10816-3)

## 6. Validar a integridade

In [12]:
from src.integracao.validacao import validar_integridade
for nome, ok, detalhe in validar_integridade(resultado["log"]):
    print("[OK]   " if ok else "[FALHA]", nome, "-", detalhe)

[OK]    Existe ao menos um evento registrado - 4 evento(s)
[OK]    Todo evento tem diagnóstico, ação e procedimentos - 0 sem recomendação
[OK]    Todo evento tem histórico de transições - 0 sem histórico
[OK]    No máximo um evento aberto por ativo - 0 ativo(s) com duplicidade
[OK]    Todo evento resolvido tem data e descrição de encerramento - 0 inconsistente(s)
[OK]    Nenhum ativo em 'atenção/crítico' sem evento aberto - 0 divergência(s) entre status e eventos
[OK]    Uma ordem de manutenção por evento (sem duplicidade) - 0 evento(s) com OS repetida
[OK]    Cadeia de hashes da auditoria íntegra - 26 registro(s) verificados: ok
[OK]    Execuções de ciclo registradas no banco - 15 ciclo(s)
[OK]    Log estruturado contém o fluxo alerta → status → recomendação - 379 linhas JSON


## 7. Testes automatizados

In [13]:
r = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q", "--color=no", "-p", "no:cacheprovider"], capture_output=True, text=True)
print(r.stdout[-700:])

..........................................                               [100%]
42 passed in 10.52s



## 8. (Opcional) Rodar com IA generativa

Com o Ollama instalado (`ollama pull llama3.2 && ollama serve`), execute novamente a demonstração. A origem das recomendações passa de `regras+rag` para `llm+rag`. Sem o Ollama, o pipeline segue funcionando, e isso está registrado no campo `origem_recomendacao` de cada evento.

## Conclusão

O pipeline operou sem intervenção manual, corrigiu o status divergente do cadastro, abriu e encerrou quatro eventos (um por técnico, três automaticamente), escalou uma falha de alerta para crítica, preservou o status de manutenção definido por pessoas e atravessou arquivos corrompidos, sensor sem sinal e falha de etapa sem interromper o fluxo.